In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn.functional as F
import torchvision.transforms as T
from PIL import Image
from torchvision.models import ResNet18_Weights, resnet18

%matplotlib inline

In [ ]:
class ResNetWrapper(torch.nn.Module):
    def __init__(self):
        super().__init__()

        weights = ResNet18_Weights.DEFAULT
        self.model = resnet18(weights=weights).eval()

        transforms = weights.transforms()
        self.normalize = T.Normalize(mean=transforms.mean, std=transforms.std)

    def forward(self, x):
        return self.model(self.normalize(x))


model = ResNetWrapper()

In [ ]:
def fgsm_attack(image, target, model, alpha):
    revert_mode = False
    if not image.requires_grad:
        revert_mode = True
        image.requires_grad_(True)

    output = model(image)
    loss = F.cross_entropy(output, target)

    model.zero_grad()
    if image.grad is not None:
        image.grad.zero_()

    loss.backward()
    image_adv = torch.clamp(image - alpha * image.grad.sign(), 0, 1)

    if revert_mode:
        image.requires_grad_(False)

    return image_adv.detach()

In [ ]:
def pgd_attack(image, target, model, alpha=0.01, epsilon=0.1, iterations=20):
    adv_image = image.clone().detach() + torch.distributions.Uniform(
        -epsilon, epsilon
    ).sample(image.shape)
    for _ in range(1, iterations + 1):
        adv_image = fgsm_attack(adv_image, target, model, alpha)

        noise = adv_image - image
        noise = torch.clamp(noise, -epsilon, epsilon)

        adv_image = torch.clamp(image + noise, 0, 1)

    return adv_image.detach()

In [ ]:
def predict_and_show(image, model, labels, ax):
    output = model(image)
    probabilities = F.softmax(output, dim=1)
    top1_prob, top1_class_idx = torch.max(probabilities, dim=1)

    ax.imshow(image.squeeze().permute(1, 2, 0).numpy())
    ax.set_title(
        f"Класс {labels[top1_class_idx.item()]} "
        f"с вероятностью {top1_prob.item():.2f}"
    )
    ax.axis("off")

    return output, probabilities

In [ ]:
resnet_transforms = ResNet18_Weights.DEFAULT.transforms()
vis_transform = T.Compose(
    [
        T.Resize(resnet_transforms.resize_size),
        T.CenterCrop(resnet_transforms.crop_size),
        T.ToTensor(),
    ]
)

categories = ResNet18_Weights.DEFAULT.meta["categories"]

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 6))

image = Image.open(Path.cwd().parent / "images" / "dog.jpg").convert("RGB")
image_tensor = vis_transform(image).unsqueeze(0)

target = torch.tensor([np.random.randint(0, len(categories))])
noise = vis_transform(
    Image.fromarray(
        np.random.normal(128, 70, (224, 224, 3)).astype(np.uint8), mode="RGB"
    )
).unsqueeze(0)
adv_noise = pgd_attack(
    noise, target, model, alpha=0.01, epsilon=0.1, iterations=5
)

predict_and_show(image_tensor, model, categories, axes[0])
predict_and_show(adv_noise, model, categories, axes[1])

plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(10, 6))

image = Image.open(Path.cwd().parent / "images" / "cow.jpg").convert("RGB")
image_tensor = vis_transform(image).unsqueeze(0)

logs, probs = predict_and_show(image_tensor, model, categories, ax)

top5_prob, top5_class_idx = torch.topk(probs, k=5, dim=1)
for i, (prob, idx) in enumerate(
    zip(top5_prob[0], top5_class_idx[0], strict=True)
):
    class_name = categories[idx.item()]
    print(
        f"{i + 1}. Класс {class_name} с вероятностью {prob:.3f} "
        f"и логитом {logs[0, idx].item()}"
    )

plt.tight_layout()
plt.show()